# P-Set 1 -- AM115 notebook starter (Parts 1 and 3)

The Part 1 and Part 3 programs from `code/*.py`, in one notebook, for anyone working in
Google Colab or Jupyter instead of a terminal. Fill in every cell marked `TODO`, run the
whole notebook top to bottom (Colab: *Runtime -> Run all*), and submit **this notebook
with its outputs showing** (Colab: *File -> Download -> Download .ipynb*) to Gradescope
**P-Set 1 code**.

**Part 2 is not here.** Its data and its specification are on the Kaggle competition page,
and the code is yours to write; submit it as `tournament_ps1.py` alongside this notebook,
with the `submission.csv` it writes.

**Data.** Run the setup cell first. Locally, put this notebook in the unpacked zip's
`code/` directory (or its top directory) and the cell finds `data/`. On Colab the cell
opens a file picker: select `ws_1905_1951.csv` from the zip. Uploads vanish when the Colab
runtime restarts, so keep the zip.

In [ ]:
import pathlib

import numpy as np

%matplotlib inline

HERE = pathlib.Path.cwd()          # outputs (nll_series.png, moments.txt) land here


def _find(name, *dirs):
    for d in dirs:
        if (HERE / d / name).exists():
            return (HERE / d).resolve()
    return None


DATA_WS_DIR = _find("ws_1905_1951.csv", ".", "data", "../data", "data/public", "../data/public")
if DATA_WS_DIR is None:
    try:
        from google.colab import files   # Colab: a file picker opens -- select ws_1905_1951.csv
    except ImportError:
        raise SystemExit("data not found: put this notebook next to data/ (see the cell above)")
    files.upload()
    DATA_WS_DIR = HERE
DEFAULT_DATA = DATA_WS_DIR / "ws_1905_1951.csv"   # Part 1
print("World Series file:", DEFAULT_DATA)

## Part 1 -- estimating p from World Series data (item 1.3)

Fill in the FOUR function bodies marked `raise NotImplementedError`: the outcome probabilities (your 1.1.a formula), the negative log-likelihood, the numerical MLE, and the series simulator. `load_counts` and `main` are supplied: read them, do not rewrite them. Keep every name, argument order and return type exactly as given. The run cell prints p-hat and the replicate standard deviations and shows `nll_series.png`.

In [ ]:
import pathlib
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.special import comb
import matplotlib
import matplotlib.pyplot as plt  # noqa: E402

In [ ]:
# TODO
def prob_loser_wins(p):
    """P(K = k | p) for k = 0..3 in a best-of-7: C(3+k,k) [p^4 q^k + q^4 p^k].

    p is the probability the BETTER team wins one game (p >= 1/2); the loser of the series
    may be either team, which is why both terms appear.
    """
    raise NotImplementedError("TODO")

In [ ]:
# TODO
def nll_series(p, counts):
    """Negative log-likelihood of the counts n_0..n_3 under P(K = k | p).

    `counts` is a length-4 array: how many series the losing team won 0, 1, 2, 3 games.
    The multinomial coefficient is dropped (it does not depend on p). Items 1.1.c and 1.3.a.
    """
    raise NotImplementedError("TODO")

In [ ]:
# TODO
def mle_series(counts):
    """Item 1.3.b: MLE of p by numerical minimisation of nll_series on [1/2, 1).

    Bounded scalar minimisation, deliberately. Item 1.1.d shows l(p) = l(1 - p), so
    dl/dp = 0 exactly at p = 1/2 -- and a gradient method such as L-BFGS-B with a lower
    bound of 0.5 can land on that bound after its first step, see a zero projected
    gradient, and report p = 0.5 as converged. (It did, from x0 = 0.75, on this data.)
    Brent's bounded method never evaluates the gradient, so it cannot be fooled that way.
    """
    raise NotImplementedError("TODO")

In [ ]:
# TODO
def simulate_series(p, n_series, rng):
    """Simulate `n_series` best-of-7 series at better-team win probability p.

    Returns the counts n_0..n_3 of how many series the losing team won 0..3 games.
    Plays each game as a Bernoulli(p) trial for the better team until one side has 4.
    """
    raise NotImplementedError("TODO")

In [ ]:
def load_counts(path):
    data = np.genfromtxt(path, delimiter=",", names=True, dtype=None, encoding="utf-8")
    return np.bincount(np.asarray(data["loser_wins"], dtype=np.intp), minlength=4)[:4]

In [ ]:
def main(path=DEFAULT_DATA):
    counts = load_counts(path)
    n = int(counts.sum())
    print(f"data: n = {n} series, counts n_0..n_3 = {counts.tolist()}")

    p_hat = mle_series(counts)
    print(f"p-hat = {p_hat:.4f}   NLL = {nll_series(p_hat, counts):.3f}   "
          f"(Mosteller: 0.65, NLL {nll_series(0.65, counts):.3f})")
    pred = n * prob_loser_wins(p_hat)
    print("predicted counts at p-hat:", np.round(pred, 2).tolist(), " observed:", counts.tolist())

    # 1.3.b -- the log-likelihood curve
    grid = np.linspace(0.5, 0.999, 400)
    ll = [-nll_series(g, counts) for g in grid]
    plt.figure(figsize=(6, 3.6))
    plt.plot(grid, ll)
    plt.axvline(p_hat, ls="--", label=f"$\\hat p$ = {p_hat:.3f}")
    plt.xlabel("p (better team's single-game win probability)")
    plt.ylabel("log-likelihood")
    plt.title("World Series 1905-1951: log-likelihood of p")
    plt.legend()
    plt.tight_layout()
    plt.savefig(HERE / "nll_series.png", dpi=120)

    # 1.3.c -- how well do 44 (and 4400) series pin p down? (interpreted in 1.4.b)
    rng = np.random.default_rng(0)
    for n_series in (44, 4400):
        est = np.array([mle_series(simulate_series(p_hat, n_series, rng)) for _ in range(2000)])
        print(f"N = {n_series:>5}: SD of p-hat over 2000 replicates = {est.std(ddof=1):.4f}  "
              f"(mean {est.mean():.4f})")

In [ ]:
main(DEFAULT_DATA)

## Part 3 -- random walks (item 3.2)

Fill in the TWO function bodies marked `raise NotImplementedError`: the walk simulator and the step variance it needs. `moments_table` is supplied and the run cell writes its output to `moments.txt`; the standard-error formulas item 3.2 quotes are in `moments_table`.

In [ ]:
import pathlib
import numpy as np

In [ ]:
# TODO
def simulate_walks(n_steps, n_walkers, delta, rng, step="pm"):
    """Positions X_0..X_n for `n_walkers` independent walks; shape (n_walkers, n_steps + 1).

    step="pm":      each step is +delta or -delta with probability 1/2
    step="uniform": each step is uniform on [-delta, delta]
    All steps are drawn in one call; there is no loop over walkers.
    """
    raise NotImplementedError("TODO")

In [ ]:
# TODO
def step_variance(delta, step):
    """Variance of ONE step: delta^2 for "pm", and your item 3.1.b answer for "uniform"."""
    raise NotImplementedError("TODO")

In [ ]:
def moments_table(delta=1.0, M=10_000, n_max=1000, checkpoints=(10, 100, 1000), seed=0):
    rng = np.random.default_rng(seed)
    lines = [f"{'walk':8s} {'n':>5s} {'mean':>8s} {'se':>7s} {'var':>9s} {'se':>8s} {'theory var':>10s}"]
    for step in ("pm", "uniform"):
        x = simulate_walks(n_max, M, delta, rng, step)
        for n in checkpoints:
            xn = x[:, n]
            mean, var = xn.mean(), xn.var(ddof=1)
            se_mean = xn.std(ddof=1) / np.sqrt(M)
            se_var = var * np.sqrt(2.0 / (M - 1))
            lines.append(f"{step:8s} {n:5d} {mean:8.3f} {se_mean:7.3f} {var:9.2f} {se_var:8.2f} "
                         f"{n * step_variance(delta, step):10.2f}")
    return "\n".join(lines)

In [ ]:
def main():
    table = moments_table()
    (HERE / "moments.txt").write_text(table + "\n")
    print(table)
    print("wrote moments.txt")

In [ ]:
main()